# Dataset Evaluation Example

이 Notebook은 **공통 평가 모듈을 호출하고, 기관별 Parser/Converter/Model 연결 코드는 Notebook에서 직접 작성**하는 예제입니다.

- `evaluator/`: 공통 제공 모듈 → 가능하면 수정하지 않음
- Notebook: 기관별 데이터/모델에 맞는 Parser·Converter·Model 연결


In [ ]:
import os

import sys

if not hasattr(sys, "get_int_max_str_digits"):
    sys.get_int_max_str_digits = lambda: 4300

if not hasattr(sys, "set_int_max_str_digits"):
    sys.set_int_max_str_digits = lambda maxdigits: None

import torch
from torch.utils.data import Dataset, DataLoader
 
from loader.data_loader import build_detection_dataset
from loader.parser import YOLOParser

## 1. 기관별 설정

기관은 자신의 Croissant metadata와 모델 경로만 지정합니다.

In [4]:
JSONLD_PATH = 'MAX_TR_DS01.jsonld'
MODEL_PATH = "yolov8_torch.pt"

SPLIT = 'test'
IMG_SIZE = (640, 640)
MAX_BOXES = 2
BATCH_SIZE = 32

device = "cuda" if torch.cuda.is_available() else "cpu"

## 2. Annotation Parser

현재 예제 Dataset은 YOLO Annotation을 사용하므로 공통으로 제공되는 `YOLOParser`를 사용합니다.

기관에서 별도 Annotation 형식을 사용하는 경우 **이 Notebook에서 직접 Parser를 구현**할 수 있습니다.

In [5]:
parser = YOLOParser(max_boxes=MAX_BOXES)

# 예: 기관 자체 Annotation 형식이라면 Notebook에서 직접 구현
# class MyParser:
#     def parse(self, annotation):
#         ...
#         return boxes, classes, num_boxes
#
# parser = MyParser()

## 3. Croissant Dataset Load

`data_loader.py`는 Croissant Dataset을 읽고 Annotation 처리를 `parser`에 위임합니다.

In [6]:
dataset = build_detection_dataset(
    jsonld_path=JSONLD_PATH,
    split=SPLIT,
    img_size=IMG_SIZE,
    max_boxes=MAX_BOXES,
    batch_size=BATCH_SIZE,
    parser=parser,
    backend="torch"
)

# element_spec 대응: 첫 배치를 꺼내 shape/dtype 을 직접 출력
images, boxes, classes, num_boxes = next(iter(dataset))

print('첫 배치 이미지:', tuple(images.shape))
print('첫 배치 박스  :', tuple(boxes.shape))
print('첫 배치 클래스:', tuple(classes.shape))
print('유효 박스 수  :', num_boxes.tolist())
print('유효 박스 수  :', num_boxes.numpy().tolist())

첫 배치 이미지: (32, 640, 640, 3)
첫 배치 박스  : (32, 2, 4)
첫 배치 클래스: (32, 2)
유효 박스 수  : [2, 2, 2, 1, 2, 2, 2, 1, 1, 2, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
유효 박스 수  : [2, 2, 2, 1, 2, 2, 2, 1, 1, 2, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


## 4. 기관별 Model Input Converter

기관은 자신의 모델 입력 규격에 맞는 변환 함수를 이 Notebook에서 직접 작성합니다.

아래는 로더가 내놓는 `(images, boxes, classes, num_boxes)` 배치를 모델(PyTorch YOLOv8)의 평가 입력 규격에 맞게 변환하는 예시입니다.
- 이미지: `(B, H, W, 3)` (NHWC) → `(B, 3, H, W)` (NCHW)
- 박스: normalized `[cx, cy, w, h]` → pixel `[x1, y1, x2, y2]`

`num_boxes`(유효 박스 수)는 패딩을 잘라내기 위해 target에 함께 실어 보냅니다.

In [7]:
def convert_for_my_model(loader, img_size, device):
    """기관 모델 입력 규격으로 변환. TF 의 dataset.map(_convert) 대응.

    각 배치: images(NCHW), {'boxes': xyxy 픽셀, 'classes': ...}
    """
    H, W = img_size

    def _convert(images, boxes, classes, num_boxes):
        imgs = images.permute(0, 3, 1, 2).contiguous().to(device)   # NHWC -> NCHW

        cx, cy, w, h = boxes.unbind(dim=-1)
        x1 = (cx - w / 2.0) * W; y1 = (cy - h / 2.0) * H
        x2 = (cx + w / 2.0) * W; y2 = (cy + h / 2.0) * H
        xyxy = torch.stack([x1, y1, x2, y2], dim=-1)                # 정규화 cxcywh -> 픽셀 xyxy

        return imgs, {"boxes": xyxy, "classes": classes, "num_boxes": num_boxes}

    for images, boxes, classes, num_boxes in loader:
        yield _convert(images, boxes, classes, num_boxes)

test_ds = convert_for_my_model(dataset, IMG_SIZE, device)

### 기관별 Converter 작성 예시

모델이 위의 예시가 아닌 경우에도 동일한 위치에서 필요한 형태로 변환하면 됩니다.

In [7]:
# 예시
# def convert_for_my_model(dataset):
#     def _convert(images, boxes, classes, num_boxes):
#         # 기관 자체 모델의 입력 규격에 맞게 변환
#         ...
#         return images, model_inputs
#     return dataset.map(_convert)
#
# test_ds = convert_for_my_model(dataset)


## 5. Model Load

In [8]:
model = torch.load(MODEL_PATH, map_location=device, weights_only=False)

## 6. Evaluation

모델과 평가 모듈은 공통 기능을 사용합니다.

In [11]:
from torchmetrics.detection import MeanAveragePrecision
from ultralytics.utils.nms import non_max_suppression

@torch.no_grad()
def evaluate_map(model, converted_loader, device,
                 conf_thres=0.001, iou_thres=0.6):
    """이미 convert_for_my_model 로 변환된 배치를 받아 mAP 계산."""
    model.to(device).eval()
    metric = MeanAveragePrecision(box_format="xyxy")

    for images, target in converted_loader:
        out = model(images)
        y = out[0] if isinstance(out, (tuple, list)) else out        # 디코딩된 예측
        dets = non_max_suppression(y, conf_thres, iou_thres, max_det=300)
        # dets[i]: (n,6) = [x1,y1,x2,y2, conf, cls]  (입력 픽셀 좌표계)

        gt_boxes, gt_classes, num_boxes = target["boxes"], target["classes"], target["num_boxes"]
        preds, targets = [], []
        for i, det in enumerate(dets):
            preds.append({"boxes":  det[:, :4].cpu(),
                          "scores": det[:, 4].cpu(),
                          "labels": det[:, 5].cpu().int()})
            n = int(num_boxes[i])                       # 정답: 유효 박스만
            targets.append({"boxes":  gt_boxes[i, :n].cpu(),
                            "labels": gt_classes[i, :n].cpu().int()})
        metric.update(preds, targets)

    return metric.compute()

In [12]:
res = evaluate_map(model, test_ds, device)

print(f"mAP@50-95 : {float(res['map']):.4f}")
print(f"mAP@50    : {float(res['map_50']):.4f}")
print(f"mAP@75    : {float(res['map_75']):.4f}")
print(f"mAR@100   : {float(res['mar_100']):.4f}")

mAP@50-95 : 0.7028
mAP@50    : 0.9297
mAP@75    : 0.7574
mAR@100   : 0.7714
